In [1]:
import json
import os
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from scipy.stats import norm
from scipy.special import expit


tfd = tfp.distributions
tfb = tfp.bijectors

In [2]:
# helpers
def binomial(n_samples, theta_like=0.7, seed=20):
    np.random.seed(seed)
    data = np.random.binomial(n_samples, theta_like)
    return data

def load_config(config_file="config.json"):

    with open(config_file, 'r') as f:
        config = json.load(f)
    return config

def make_conditioned_lp(prior_dist, likelihood_dist, x):
    if type(x) is not int and (x is None or len(x) == 0):
        return lambda z: prior_dist.log_prob(z)
    else:
        def log_prob_fn(z):
            # z: (sample_size,)
            # x: (n_data,)
            z_exp = tf.expand_dims(z, -1)        # (sample_size, 1)
            x_exp = tf.expand_dims(x, 0)         # (1, n_data)
            lp = prior_dist.log_prob(z)          # (sample_size,)
            lp += tf.reduce_sum(likelihood_dist(z_exp).log_prob(x_exp), axis=-1)  # (sample_size,)
            return lp
        return log_prob_fn

def _logistic_normal_mean_std_mc(mu, sigma, n_samples=10_000, seed=0):
    mu = float(np.squeeze(mu))
    sigma = float(np.squeeze(sigma))
    rng = np.random.default_rng(seed)
    z = rng.normal(loc=mu, scale=sigma, size=n_samples)
    theta = expit(z)
    return float(theta.mean()), float(theta.std(ddof=1))

def _trace_locscale_to_theta_moments(loc_trace, scale_trace, n_samples=10_000, seed=0):
    loc_trace = np.asarray(loc_trace, dtype=float)
    scale_trace = np.asarray(scale_trace, dtype=float)
    out_mean = np.empty(loc_trace.shape[0], dtype=float)
    out_std  = np.empty(loc_trace.shape[0], dtype=float)
    for i in range(loc_trace.shape[0]):
        out_mean[i], out_std[i] = _logistic_normal_mean_std_mc(
            loc_trace[i], scale_trace[i], n_samples=n_samples, seed=seed
        )
    return out_mean, out_std

data_gen = binomial

In [3]:
config_file = "../beta_config.json"
config = load_config(config_file)

theta_like = config['theta_like']
alpha_prior = config['alpha_prior']
beta_prior = config['beta_prior']
n_samples = config['n_samples']
grad_samps = config['grad_samps']
max_iters = 100_000
adam_step = config['adam_step']
transformation = config['transformation']


data = data_gen(n_samples) if n_samples > 0 else 0

In [4]:
# copy-pasted from beta_tfp.ipynb
def logistic_normal_mean_std_mc(mu, sigma, n_samples=10_000, seed=1):
    mu = float(np.squeeze(mu))
    sigma = float(np.squeeze(sigma))
    rng = np.random.default_rng(seed)
    z = rng.normal(loc=mu, scale=sigma, size=n_samples)
    theta = expit(z)  # map to (0,1)
    mean = float(theta.mean())
    std = float(theta.std(ddof=1))
    return mean, std

def make_conditioned_lp_binomial_count(alpha_prior, beta_prior, total_count, y_obs, dtype=tf.float32):
    alpha = tf.convert_to_tensor(alpha_prior, dtype=dtype)
    beta  = tf.convert_to_tensor(beta_prior,  dtype=dtype)

    # Cast to float for TFP Binomial internal computations (avoids int32/float32 mismatch errors)
    total_count_f = tf.cast(total_count, dtype)
    y_obs_f       = tf.cast(y_obs, dtype)

    prior = tfd.Beta(concentration1=alpha, concentration0=beta)

    @tf.function(jit_compile=False)  # keep simple; VI can be jit-compiled separately
    def log_prob_fn(theta):
        theta = tf.convert_to_tensor(theta, dtype=dtype)  # theta in (0,1), vector shape [sample_size]
        lp = prior.log_prob(theta)
        lp += tfd.Binomial(total_count=total_count_f, probs=theta).log_prob(y_obs_f)
        return lp

    return log_prob_fn


def build_surrogate(init_params=None):
    init_dict = None
    if init_params is not None:
        init_loc, init_scale = init_params
        init_dict = {
            "loc": tf.convert_to_tensor(init_loc, dtype=tf.float32),
            "scale": tf.convert_to_tensor(init_scale, dtype=tf.float32),
        }

    q = tfp.experimental.vi.build_factored_surrogate_posterior(
        event_shape=(),
        bijector=tfb.Sigmoid(),         # theta = sigmoid(z)
        initial_parameters=init_dict,   # loc/scale of base Normal z
    )
    return q


def make_trace_fn(q):
    def trace_fn(_traceable_quantities):
        base = q.distribution  # Normal(loc, scale) on R (pre-sigmoid)
        return base.loc, base.scale
    return trace_fn


alpha_prior = float(alpha_prior)
beta_prior  = float(beta_prior)
n_samples   = int(n_samples)
data        = int(data)

conditioned_log_prob = make_conditioned_lp_binomial_count(
    alpha_prior=alpha_prior,
    beta_prior=beta_prior,
    total_count=n_samples,
    y_obs=data,
    dtype=tf.float32
)

def run_restart(seed):
    tf.random.set_seed(seed)

    single_q = build_surrogate()
    multi_q  = build_surrogate()

    single_trace_fn = make_trace_fn(single_q)
    multi_trace_fn  = make_trace_fn(multi_q)


    single_trace = tfp.vi.fit_surrogate_posterior(
        target_log_prob_fn=conditioned_log_prob,
        surrogate_posterior=single_q,
        trainable_variables=single_q.trainable_variables,
        optimizer=tf.optimizers.Adam(),
        num_steps=max_iters,
        seed=seed,
        sample_size=1,
        trace_fn=single_trace_fn,
        jit_compile=True,
    )

    multi_trace = tfp.vi.fit_surrogate_posterior(
        target_log_prob_fn=conditioned_log_prob,
        surrogate_posterior=multi_q,
        trainable_variables=multi_q.trainable_variables,
        optimizer=tf.optimizers.Adam(),
        num_steps=max_iters,
        seed=seed+1000,
        sample_size=100,
        trace_fn=multi_trace_fn,
        jit_compile=True,
    )

    single_loc_trace, single_scale_trace = single_trace
    multi_loc_trace,  multi_scale_trace  = multi_trace
    return single_loc_trace, single_scale_trace, multi_loc_trace, multi_scale_trace

In [6]:
results = []
for seed in tqdm(range(100), position=1, desc="Random restarts..."):
    result = run_restart(seed)
    results.append(result)

Random restarts...:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
plt.rcParams.update({'font.size': 20})
# ----------------------------
# unpack results -> stacks also transform
# ----------------------------
single_means, single_stds = [], []
multi_means,  multi_stds  = [], []
for single_loc_trace, single_scale_trace, multi_loc_trace, multi_scale_trace in tqdm(results):
    # Use vectorized transformation for speed
    single_mu, single_sigma = _trace_locscale_to_theta_moments(
        single_loc_trace, single_scale_trace, n_samples=10_000, seed=0
    )
    multi_mu, multi_sigma = _trace_locscale_to_theta_moments(
        multi_loc_trace, multi_scale_trace, n_samples=10_000, seed=0
    )
    
    single_means.append(single_mu)
    single_stds.append(single_sigma)
    multi_means.append(multi_mu)
    multi_stds.append(multi_sigma)

single_means = np.stack(single_means, axis=0)  # (N, T)
single_stds  = np.stack(single_stds,  axis=0)  # (N, T)
multi_means  = np.stack(multi_means,  axis=0)  # (N, T)
multi_stds   = np.stack(multi_stds,   axis=0)  # (N, T)

N, T = single_means.shape
x = np.arange(T)

# If you have "best" from config, these should be on theta-scale (0,1)
best_mu = float(config["best_mean"]) if "config" in globals() and "best_mean" in config else None
best_std = float(config["best_std"]) if "config" in globals() and "best_std" in config else None
best_mu, best_std = _logistic_normal_mean_std_mc(best_mu, best_std, n_samples=50_000, seed=1)
# ----------------------------
# Plot 1: overlay a few trajectories from each condition
# ----------------------------
k = 2  # how many trajectories to show from each
idx = np.linspace(0, N - 1, k, dtype=int)  # deterministic selection

fig, axs = plt.subplots(2, 1, figsize=(12, 14))

# std of mu
for i in idx:
    axs[0].plot(x, single_stds[i], alpha=0.6, linewidth=1, color='blue')
for i in idx:
    axs[0].plot(x, multi_stds[i],  alpha=0.6, linewidth=1, color='green')

axs[0].axhline(best_std, color='red', linestyle='--', label=r'Best Variational approx of $\sigma_p$')
axs[0].set_title(r'Variational Approximation of $\sigma_p$ (Posterior Std of $\mu$): a few runs')
axs[0].set_xlabel('Iteration')
axs[0].set_ylabel(r'$\sigma_p$')
axs[0].grid()
axs[0].set_ylim(best_std * 0.6, best_std * 1.6)

# mean of mu
for i in idx:
    axs[1].plot(x, single_means[i], alpha=0.6, linewidth=1, color='blue', label=None)
for i in idx:
    axs[1].plot(x, multi_means[i],  alpha=0.6, linewidth=1, label=None)

axs[1].axhline(best_mu, color='red', linestyle='--', label=r'Best Variational approx of $\mu_p$')
axs[1].set_title(r'Variational Approximation of $\mu_p$ (Posterior Mean of $\mu$): a few runs')
axs[1].set_xlabel('Iteration')
axs[1].set_ylabel(r'$\mu_p$')
axs[1].grid()
axs[1].set_ylim(best_mu - 3 * best_std, best_mu + 3 * best_std)

# manual legend (so we don’t get 16 duplicate entries)
axs[0].plot([], [], color='blue', label='1 MC sample (some runs)')
axs[0].plot([], [], color='green', label='100 MC samples (some runs)')
axs[0].legend()

axs[1].plot([], [], color='blue', label='1 MC sample (some runs)')
axs[1].plot([], [], color='green', label='100 MC samples (some runs)')
axs[1].legend()

plt.tight_layout()
plt.show()

# ----------------------------
# Plot 2: mean trajectory ± 1 std band across runs (separate plots for 1-MC and 100-MC)
# ----------------------------
def plot_mean_band(ax, x, Y, true_value, title, ylabel):
    """
    Y: (N, T) trajectories
    """
    m = Y.mean(axis=0)
    s = Y.std(axis=0)
    ax.plot(x, m, linewidth=2, label='Mean across runs')
    ax.fill_between(x, m - s, m + s, alpha=0.25)
    ax.axhline(true_value, color='red', linestyle='--', label='Best value')
    ax.set_title(title)
    ax.set_xlabel('Iteration')
    ax.set_ylabel(ylabel)
    ax.grid()
    ax.legend()

# 1 MC sample: mean ± sd across runs
fig, axs = plt.subplots(2, 1, figsize=(12, 14))
plot_mean_band(
    axs[0], x, single_stds, best_std,
    r'1 MC sample: $\sigma_p$ mean trajectory $\pm$ 1 SD across runs',
    r'$\sigma_p$'
)
axs[0].set_ylim(best_std * 0.6, best_std * 1.6)

plot_mean_band(
    axs[1], x, single_means, best_mu,
    r'1 MC sample: $\mu_p$ mean trajectory $\pm$ 1 SD across runs',
    r'$\mu_p$'
)
axs[1].set_ylim(best_mu - 3 * best_std, best_mu + 3 * best_std)

plt.tight_layout()
plt.show()

# 100 MC samples: mean ± sd across runs
fig, axs = plt.subplots(2, 1, figsize=(12, 14))
plot_mean_band(
    axs[0], x, multi_stds, best_std,
    r'100 MC samples: $\sigma_p$ mean trajectory $\pm$ 1 SD across runs',
    r'$\sigma_p$'
)
axs[0].set_ylim(best_std * 0.6, best_std * 1.6)

plot_mean_band(
    axs[1], x, multi_means, best_mu,
    r'100 MC samples: $\mu_p$ mean trajectory $\pm$ 1 SD across runs',
    r'$\mu_p$'
)
axs[1].set_ylim(best_mu - 3 * best_std, best_mu + 3 * best_std)

plt.tight_layout()
plt.show()


  0%|          | 0/100 [00:00<?, ?it/s]